# Lab 13 — Dataset Curation Pipeline
**Goal:** Create a miniature document pipeline with filtering, provenance, and quality fields.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
docs=[
{"id":1,"source":"web","lang":"en","text":"A clean technical document about machine learning."},
{"id":2,"source":"web","lang":"en","text":"BUY NOW!!! click click click"},
{"id":3,"source":"book","lang":"fa","text":"این یک متن فارسی آموزشی است."},
{"id":4,"source":"unknown","lang":"en","text":""},
]
print(len(docs))


## Prediction
What metadata would you refuse to lose when building a serious corpus?


In [ ]:
def quality(d):
    txt=d["text"].strip()
    return {"nonempty":bool(txt),"len":len(txt),"has_spam":("buy now" in txt.lower() or "click click" in txt.lower())}
for d in docs:
    d["quality"]=quality(d)
    print(d)


## Interpretation
At minimum: source/provenance, license or access basis, language, document ID, filtering decisions, deduplication identifiers, and dataset version.


In [ ]:
kept=[d for d in docs if d["quality"]["nonempty"] and not d["quality"]["has_spam"]]
print("kept:",len(kept))
print("languages:",{d["lang"] for d in kept})
print("sources:",{d["source"] for d in kept})


## Answers
Curation is an experimental pipeline. Without provenance, you cannot explain changes in model quality, reproduce a dataset, or perform credible audits.

### Primary source
https://arxiv.org/abs/2402.00159


## Deepening: quantify every transformation

This section upgrades the notebook from a runnable demo to a measurable experiment. Predict before executing.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
import re, hashlib
from collections import Counter

docs = [
    {"id":"a","lang":"en","text":"Language models learn statistical patterns."},
    {"id":"b","lang":"en","text":"Language models learn statistical patterns."},
    {"id":"c","lang":"ur","text":"یہ ایک مثال ہے۔"},
    {"id":"d","lang":"en","text":"Useful data should be measured and versioned."},
]

def tok(s): return re.findall(r"\S+", s)

normalized = [{**d, "text":" ".join(d["text"].split())} for d in docs]
seen, deduped = set(), []
for d in normalized:
    h = hashlib.sha256(d["text"].encode()).hexdigest()
    if h not in seen:
        seen.add(h); deduped.append(d)

for name, ds in [("raw", docs), ("normalized", normalized), ("deduped", deduped)]:
    c = Counter(d["lang"] for d in ds)
    t = sum(len(tok(d["text"])) for d in ds)
    print(name, "docs=", len(ds), "tokens=", t, "langs=", dict(c))


## Break experiment

Add a language-specific filter that accidentally removes every non-English example.

Predict overall and per-language survival. Then fix it.

### Report

source → transformation → survival rate → language effect → correction

Use templates/EXPERIMENT_CARD.md to record the run.
